In [2]:
!pip install geopy

   ---------------------------------------- 0.0/125.4 kB ? eta -:--:--
   ---------------------------------------- 125.4/125.4 kB 3.7 MB/s eta 0:00:00
   ---------------------------------------- 0.0/40.7 kB ? eta -:--:--
   ---------------------------------------- 40.7/40.7 kB 1.9 MB/s eta 0:00:00


In [3]:
# save as generate_mumbai_data.py and run: python generate_mumbai_data.py
import random
import math
import pandas as pd
from datetime import datetime, timedelta
import requests
import time
from geopy.geocoders import Nominatim
from tqdm import tqdm

In [50]:
# ---------- CONFIG ----------
NUM_ROWS = 100_000   # change if you want more/fewer
OUTPUT_CSV = "mumbai_dynamic_pricing_100k.csv"

# Optional: use real OpenWeather (recommended to call only once or few times to avoid rate limits)
USE_REAL_WEATHER = False
OPENWEATHER_API = "d9f1aaa8d7df65d34675da384b1840ec"  # put your key or leave blank if USE_REAL_WEATHER=False

# Mumbai bounding box (approx)
# lat_min, lat_max, lon_min, lon_max
MUMBAI_BBOX = (18.8920, 19.2710, 72.7750, 72.9860)

# Pricing parameters (example, tune as needed)
BASE_FARE = 30.0      # base fare in INR
PER_KM = 12.0         # per km charge
PER_MIN = 2.0         # per minute charge

# Simulated speeds (km/h) by hour type and weather
BASE_SPEED_KMPH = 30.0   # average speed
RUSH_HOUR_SPEED = 18.0
RAIN_SPEED_FACTOR = 0.7  # multiply base speed by this in rain

# Rush hours set
RUSH_HOURS = set([8,9,17,18,19])

# Helper functions
def random_point_in_bbox(bbox):
    lat = random.uniform(bbox[0], bbox[1])
    lon = random.uniform(bbox[2], bbox[3])
    return lat, lon

def haversine_distance_km(lat1, lon1, lat2, lon2):
    # Haversine formula
    R = 6371.0  # Earth radius km
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2.0)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2.0)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

def fetch_mumbai_weather_once():
    # Fetch current weather for Mumbai via OpenWeather (if key provided)
    if not OPENWEATHER_API:
        return None
    try:
        url = f"https://api.openweathermap.org/data/2.5/weather?q=Mumbai&appid={OPENWEATHER_API}&units=metric"
        r = requests.get(url, timeout=10)
        data = r.json()
        # minimal checks
        if "main" in data:
            return {
                "temp": data["main"]["temp"],
                "humidity": data["main"].get("humidity", None),
                "weather": data["weather"][0]["main"] if data.get("weather") else "Clear"
            }
    except Exception as e:
        print("Weather fetch error:", e)
    return None

# Optionally get one weather snapshot for Mumbai
weather_snapshot = None
if USE_REAL_WEATHER and OPENWEATHER_API:
    print("Fetching Mumbai weather...")
    weather_snapshot = fetch_mumbai_weather_once()
    print("Weather snapshot:", weather_snapshot)
    time.sleep(1)

# Prepare to generate rows
rows = []
start_time = datetime.now()

for i in range(NUM_ROWS):
    # 1. Random pickup and drop inside Mumbai bounding box
    p_lat, p_lon = random_point_in_bbox(MUMBAI_BBOX)
    d_lat, d_lon = random_point_in_bbox(MUMBAI_BBOX)
    
    # Avoid extremely tiny trips (~same point); ensure distance >= 0.3 km
    distance_km = haversine_distance_km(p_lat, p_lon, d_lat, d_lon)
    if distance_km < 0.2:
        # regenerate a drop point once (simple)
        d_lat, d_lon = random_point_in_bbox(MUMBAI_BBOX)
        distance_km = haversine_distance_km(p_lat, p_lon, d_lat, d_lon)
    
    # 2. Simulate time features
    # Random date-time in last 30 days (helps add variety)
    random_minutes_ago = random.randint(0, 30*24*60)
    timestamp = datetime.now() - timedelta(minutes=random_minutes_ago)
    hour = timestamp.hour
    day_of_week = timestamp.weekday()  # 0=Mon,6=Sun
    is_weekend = 1 if day_of_week >= 5 else 0
    
    # 3. Determine weather (use snapshot if available else simulate)
    if weather_snapshot:
        weather = weather_snapshot["weather"]
        temp = weather_snapshot["temp"]
    else:
        # simulate weather categories with probabilities
        # Most days: Clear or Clouds; rain sometimes
        weather_choice = random.choices(
            ["Clear", "Clouds", "Rain", "Thunderstorm"],
            weights=[0.5, 0.3, 0.15, 0.05],
            k=1
        )[0]
        weather = weather_choice
        # temp roughly between 20-35 with some noise
        temp = round(random.uniform(24, 33), 1)

    # 4. Speed estimation (km/h) based on hour & weather
    avg_speed = BASE_SPEED_KMPH
    if hour in RUSH_HOURS:
        avg_speed = RUSH_HOUR_SPEED
    if weather in ["Rain", "Thunderstorm"]:
        avg_speed = avg_speed * RAIN_SPEED_FACTOR

    # 5. Estimate duration (minutes)
    # duration = distance / speed (hours) * 60
    duration_min = (distance_km / max(avg_speed, 1)) * 60.0
    # add some noise
    duration_min = duration_min * random.uniform(0.9, 1.2)
    
    # 6. Simulate demand and supply
    # demand higher in rush hours & bad weather
    base_demand = random.randint(50, 200)
    if hour in RUSH_HOURS:
        base_demand += random.randint(100, 400)
    if weather in ["Rain", "Thunderstorm"]:
        base_demand += random.randint(50, 200)
    # supply somewhat lower during rush hours / bad weather
    base_supply = random.randint(30, 250)
    if hour in RUSH_HOURS:
        base_supply = max(10, base_supply - random.randint(20, 80))
    if weather in ["Rain", "Thunderstorm"]:
        base_supply = max(5, base_supply - random.randint(20, 100))
    
    demand_supply_ratio = base_demand / max(base_supply, 1)
    
    # 7. Surge factor logic (simple but realistic)
    surge = 1.0
    # baseline surge from demand/supply
    if demand_supply_ratio > 2.0:
        surge += min(1.5, (demand_supply_ratio - 1.0) * 0.15)  # cap
    elif demand_supply_ratio > 1.2:
        surge += (demand_supply_ratio - 1.0) * 0.08
    # add rush hour and weather bump
    if hour in RUSH_HOURS:
        surge += 0.1
    if weather in ["Rain", "Thunderstorm"]:
        surge += 0.2
    # small random noise
    surge = round(surge * random.uniform(0.95, 1.05), 2)

    # 8. Price computation
    price = (BASE_FARE + PER_KM * distance_km + PER_MIN * duration_min) * surge
    price = round(price, 2)

    row = {
        "pickup_lat": round(p_lat, 6),
        "pickup_lon": round(p_lon, 6),
        "drop_lat": round(d_lat, 6),
        "drop_lon": round(d_lon, 6),
        "distance_km": round(distance_km, 3),
        "duration_min": round(duration_min, 2),
        "hour": hour,
        "day_of_week": day_of_week,
        "is_weekend": is_weekend,
        "temperature_C": temp,
        "weather": weather,
        "demand": base_demand,
        "supply": base_supply,
        "demand_supply_ratio": round(demand_supply_ratio, 2),
        "surge_factor": surge,
        "price_inr": price,
        "timestamp": timestamp.strftime("%Y-%m-%d %H:%M:%S")
    }
    rows.append(row)

    # optional: simple progress print
    if (i+1) % 10000 == 0:
        print(f"{i+1} rows generated...")

# save
df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False)
print("✅ Done. Saved to:", OUTPUT_CSV)
print("Rows:", len(df))


10000 rows generated...
20000 rows generated...
30000 rows generated...
40000 rows generated...
50000 rows generated...
60000 rows generated...
70000 rows generated...
80000 rows generated...
90000 rows generated...
100000 rows generated...
✅ Done. Saved to: mumbai_dynamic_pricing_100k.csv
Rows: 100000


In [52]:
df

,pickup_lat,pickup_lon,drop_lat,drop_lon,distance_km,duration_min,hour,day_of_week,is_weekend,temperature_C,weather,demand,supply,demand_supply_ratio,surge_factor,price_inr,timestamp
0,19.158700,72.922719,19.244828,72.854938,11.932,43.92,18,6,1,26.9,Clouds,340,227,1.50,1.19,310.62,2025-10-12 18:51:29
1,18.902577,72.836593,19.085842,72.808627,20.589,55.54,22,6,1,32.9,Thunderstorm,232,82,2.83,1.48,574.47,2025-10-26 22:47:29
2,19.254640,72.790320,18.970944,72.928854,34.742,80.91,0,0,0,24.5,Clouds,167,100,1.67,1.02,620.90,2025-10-20 00:24:29
3,19.203422,72.973388,18.952843,72.780283,34.470,112.66,12,4,0,26.9,Rain,180,16,11.25,2.69,1799.48,2025-10-17 12:07:29
4,19.086513,72.938214,19.151382,72.825271,13.886,28.15,2,0,0,32.5,Clear,137,220,0.62,1.03,260.52,2025-10-20 02:59:29
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,19.231268,72.961815,18.905969,72.883407,37.098,76.93,5,0,0,30.8,Clouds,64,127,0.50,1.03,647.92,2025-10-13 05:54:33
99996,18.904766,72.826730,19.213805,72.856480,34.506,67.24,1,4,0,29.9,Clear,96,162,0.59,0.95,549.62,2025-10-17 01:32:33
99997,19.231143,72.917236,19.152966,72.786397,16.259,58.72,9,0,0,24.2,Clouds,256,53,4.83,1.73,592.60,2025-10-27 09:38:33
99998,19.068336,72.934359,19.059075,72.883201,5.474,10.22,2,2,0,32.5,Clouds,120,194,0.62,0.98,113.81,2025-11-05 02:18:33


In [5]:
pip install geopy tqdm

Note: you may need to restart the kernel to use updated packages.


In [4]:
df = pd.read_csv("mumbai_dynamic_pricing_100k.csv")

In [6]:
df

,pickup_lat,pickup_lon,drop_lat,drop_lon,distance_km,duration_min,hour,day_of_week,is_weekend,temperature_C,weather,demand,supply,demand_supply_ratio,surge_factor,price_inr,timestamp
0,19.047378,72.831144,18.990603,72.843779,6.451,13.97,13,6,1,28.6,Clear,123,133,0.92,1.02,138.06,2025-10-26 13:14:36
1,19.144254,72.805980,18.938734,72.911354,25.395,85.86,1,6,1,32.5,Rain,247,132,1.87,1.29,653.33,2025-10-12 01:32:36
2,18.945738,72.896440,18.959863,72.880630,2.287,8.70,8,5,1,26.1,Clear,517,166,3.11,1.48,110.79,2025-11-01 08:41:36
3,18.997853,72.931312,19.186420,72.908905,21.099,115.36,8,4,0,31.9,Thunderstorm,431,116,3.72,1.78,914.78,2025-10-24 08:02:36
4,19.253969,72.926062,19.077080,72.820992,22.554,50.86,21,0,0,31.7,Clear,51,169,0.30,1.01,406.39,2025-10-20 21:50:36
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,18.979503,72.944902,18.896560,72.857304,13.036,29.93,6,0,0,32.0,Clear,53,89,0.60,0.96,236.45,2025-11-03 06:54:40
99996,19.091841,72.792585,18.940563,72.859403,18.229,40.65,5,4,0,24.8,Clouds,95,85,1.12,1.00,330.04,2025-10-17 05:37:40
99997,19.034633,72.848680,19.255699,72.930668,26.047,86.95,9,0,0,26.9,Clear,294,121,2.43,1.25,645.56,2025-10-13 09:48:40
99998,19.153826,72.907321,19.139875,72.870808,4.137,8.89,22,6,1,25.3,Clouds,108,228,0.47,1.02,99.37,2025-10-19 22:29:40


In [7]:
geolocator = Nominatim(user_agent="mumbai_pricing_project")

In [ ]:
import pandas as pd
from geopy.geocoders import Nominatim
import time
from tqdm import tqdm

# -------------------------
# 1️⃣ Load your dataset
# -----------------------

# -------------------------
# 3️⃣ Helper function to get location
# -------------------------
def get_location(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), language='en')
        address = location.address if location else None
        time.sleep(1)  # respect rate limit (1 request/sec)
        return address
    except Exception as e:
        print(f"Error for ({lat}, {lon}): {e}")
        time.sleep(1)
        return None

# -------------------------
# 4️⃣ Create unique coordinate sets to avoid repeated lookups
# -------------------------
pickup_coords = df[['pickup_lat', 'pickup_lon']].drop_duplicates()
drop_coords = df[['drop_lat', 'drop_lon']].drop_duplicates()

print(f"Unique pickup points: {len(pickup_coords)}")
print(f"Unique drop points: {len(drop_coords)}")

# -------------------------
# 5️⃣ Reverse geocode pickups
# -------------------------
tqdm.pandas(desc="Geocoding pickup locations")
pickup_coords['pickup_location'] = pickup_coords.progress_apply(
    lambda x: get_location(x['pickup_lat'], x['pickup_lon']), axis=1
)

# -------------------------
# 6️⃣ Reverse geocode drops
# -------------------------
tqdm.pandas(desc="Geocoding drop locations")
drop_coords['drop_location'] = drop_coords.progress_apply(
    lambda x: get_location(x['drop_lat'], x['drop_lon']), axis=1
)

# -------------------------
# 7️⃣ Merge back to original DataFrame
# -------------------------
df = df.merge(pickup_coords, on=['pickup_lat', 'pickup_lon'], how='left')
df = df.merge(drop_coords, on=['drop_lat', 'drop_lon'], how='left')

# -------------------------
# 8️⃣ Save final result
# -------------------------
output_file = "rides_with_locations_large.csv"
df.to_csv(output_file, index=False, encoding='utf-8')

print(f"\n✅ Done! File saved as: {output_file}")


Unique pickup points: 100000
Unique drop points: 100000


Geocoding pickup locations:   0%|                                               | 65/100000 [01:37<40:14:19,  1.45s/it]

In [15]:

def reverse_geocode(lat, lon):
    try:
        location = geolocator.reverse((lat, lon), exactly_one=True, language="en")
        if location and "address" in location.raw:
            addr = location.raw["address"]
            return addr.get("suburb") or addr.get("neighbourhood") or addr.get("city_district") or addr.get("city") or "Unknown"
        return "Unknown"
    except Exception:
        return "Error"

In [17]:
tqdm.pandas()
df_sample = df.head(200)  # test on 200 first
df_sample["pickup_location"] = df_sample.progress_apply(lambda x: reverse_geocode(x["pickup_lat"], x["pickup_lon"]), axis=1)
df_sample["drop_location"] = df_sample.progress_apply(lambda x: reverse_geocode(x["drop_lat"], x["drop_lon"]), axis=1)

100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [05:38<00:00,  1.69s/it]
C:\Users\Rahul\AppData\Local\Temp\ipykernel_9224\3991666204.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_sample["pickup_location"] = df_sample.progress_apply(lambda x: reverse_geocode(x["pickup_lat"], x["pickup_lon"]), axis=1)
100%|████████████████████████████████████████████████████████████████████████████████| 200/200 [04:09<00:00,  1.25s/it]
C:\Users\Rahul\AppData\Local\Temp\ipykernel_9224\3991666204.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/

In [19]:
df_sample.to_csv("mumbai_with_locations_sample.csv", index=False)

In [29]:
df_sample.shape

(200, 19)

In [31]:
import random

areas = ["Andheri", "Bandra", "Powai", "Dadar", "Borivali", "Colaba", "Kurla", "Goregaon", "Thane", "Vashi"]

# Fix: use .loc to assign safely
df_sample.loc[:, "pickup_location"] = [random.choice(areas) for _ in range(len(df_sample))]
df_sample.loc[:, "drop_location"] = [random.choice(areas) for _ in range(len(df_sample))]


In [33]:
df_sample

,pickup_lat,pickup_lon,drop_lat,drop_lon,distance_km,duration_min,hour,day_of_week,is_weekend,temperature_C,weather,demand,supply,demand_supply_ratio,surge_factor,price_inr,timestamp,pickup_location,drop_location
0,19.047378,72.831144,18.990603,72.843779,6.451,13.97,13,6,1,28.6,Clear,123,133,0.92,1.02,138.06,2025-10-26 13:14:36,Dadar,Andheri
1,19.144254,72.805980,18.938734,72.911354,25.395,85.86,1,6,1,32.5,Rain,247,132,1.87,1.29,653.33,2025-10-12 01:32:36,Dadar,Andheri
2,18.945738,72.896440,18.959863,72.880630,2.287,8.70,8,5,1,26.1,Clear,517,166,3.11,1.48,110.79,2025-11-01 08:41:36,Powai,Dadar
3,18.997853,72.931312,19.186420,72.908905,21.099,115.36,8,4,0,31.9,Thunderstorm,431,116,3.72,1.78,914.78,2025-10-24 08:02:36,Thane,Vashi
4,19.253969,72.926062,19.077080,72.820992,22.554,50.86,21,0,0,31.7,Clear,51,169,0.30,1.01,406.39,2025-10-20 21:50:36,Andheri,Borivali
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,18.933098,72.880226,18.993547,72.814844,9.615,18.46,22,2,0,26.2,Clouds,166,178,0.93,0.97,176.83,2025-10-15 22:47:36,Powai,Thane
196,19.017328,72.789782,18.957133,72.839646,8.502,15.52,23,0,0,26.7,Clear,144,121,1.19,1.00,163.08,2025-10-20 23:55:36,Goregaon,Andheri
197,19.035542,72.880716,19.156643,72.941898,14.922,50.30,21,4,0,30.2,Thunderstorm,157,170,0.92,1.17,362.31,2025-10-24 21:57:36,Colaba,Thane
198,19.154497,72.819103,19.055097,72.877109,12.622,68.56,8,6,1,25.0,Thunderstorm,576,99,5.82,2.10,669.04,2025-10-19 08:18:36,Vashi,Andheri


In [35]:
def map_area(lat, lon):
    if 19.00 <= lat <= 19.10 and 72.80 <= lon <= 72.90:
        return "Andheri"
    elif 19.05 <= lat <= 19.15 and 72.95 <= lon <= 73.00:
        return "Powai"
    elif 18.95 <= lat <= 19.05 and 72.80 <= lon <= 72.90:
        return "Bandra"
    elif 19.10 <= lat <= 19.20 and 72.85 <= lon <= 72.95:
        return "Goregaon"
    else:
        return "Unknown"

df["pickup_location"] = df.apply(lambda x: map_area(x["pickup_lat"], x["pickup_lon"]), axis=1)
df["drop_location"] = df.apply(lambda x: map_area(x["drop_lat"], x["drop_lon"]), axis=1)
